# Laboratorio 6 — Evaluar un clasificador

El modelo logístico del laboratorio anterior ya está ajustado. Este laboratorio no
lo vuelve a entrenar: parte de los **puntajes** que asigna a las 1200 solicitudes
del conjunto de prueba y construye, sin biblioteca de métricas, las herramientas
para evaluarlo como clasificador:

1. La **matriz de confusión** y las cinco **métricas de umbral fijo**.
2. El barrido que produce la curva **ROC** y la curva **precision-recall**.
3. El **AUC** por conteo de pares y la **average precision**.
4. El **umbral óptimo** bajo tres matrices de costos distintas.

Hay **seis ejercicios**. Las celdas que contienen `raise NotImplementedError()` deben
completarse; el resto es andamiaje y se ejecuta sin modificar.

## 0. Preparación

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                              recall_score, f1_score, roc_auc_score,
                              average_precision_score)

plt.rcParams.update({"figure.figsize": (8, 3.6), "axes.grid": True, "grid.alpha": 0.3})
np.set_printoptions(precision=4, suppress=True)

AZUL, TURQUESA, NARANJA = "#19326E", "#50ACB0", "#CD742A"
VIOLETA, OLIVA, AZUL_CLARO = "#A3477F", "#89A943", "#4294CC"

In [ ]:
from pathlib import Path
import urllib.request

# Carga de datos del curso. No hace falta modificar nada.
URL_DATOS = "https://raw.githubusercontent.com/gustavovazquez/datasets/main/"


def ruta_del_dato(nombre: str) -> str:
    """Devuelve algo que `pd.read_csv` pueda leer, este donde este el notebook.

    Busca el archivo junto al notebook y en las carpetas `data/` de hasta tres
    niveles superiores; si no esta, usa la copia publicada en linea; y si
    tampoco hay red, en Colab ofrece subirlo a mano.
    """
    candidatas = [Path(nombre), Path("data") / nombre]
    for arriba in ("..", "../..", "../../.."):
        candidatas += [Path(arriba) / nombre, Path(arriba) / "data" / nombre]
    for ruta in candidatas:
        if ruta.exists():
            print(f"Cargado desde {ruta}")
            return str(ruta)

    url = URL_DATOS + nombre
    try:
        urllib.request.urlopen(url, timeout=15)
        print(f"No esta en disco; se descarga de {url}")
        return url
    except Exception:
        pass

    try:
        from google.colab import files  # type: ignore
    except ImportError as exc:
        raise FileNotFoundError(
            f"No se encontro {nombre}. Descargalo del repositorio del curso "
            f"y dejalo junto al notebook, o en una carpeta data/."
        ) from exc
    print(f"Falta {nombre}: seleccionalo para subirlo.")
    files.upload()
    return nombre


### Los datos y el modelo

El conjunto es el mismo de la regresión logística: 4000 solicitudes de crédito de
consumo, con `incumplio = 1` como clase positiva. Se lo particiona de manera
estratificada en 2800 solicitudes de entrenamiento y **1200 de prueba**, y el
modelo se ajusta sobre las primeras. **Todo lo que sigue se calcula sobre las
segundas**, que el ajuste no vio.

In [ ]:
#@title Carga del conjunto de datos (ejecutar sin modificar) { display-mode: "form" }
datos = pd.read_csv(ruta_del_dato("riesgo_credito.csv"))
print(f"{datos.shape[0]} filas x {datos.shape[1]} columnas")
print(f"Tasa de incumplimiento: {datos['incumplio'].mean():.1%}")
datos.head(3)

In [ ]:
#@title Matriz de atributos, partición y ajuste (ejecutar sin modificar) { display-mode: "form" }
# Misma matriz de diseño del laboratorio anterior: "al dia" como categoria de
# referencia, el monto en logaritmo y la edad centrada.
tabla = datos.copy()
tabla["hist_atrasos"] = (tabla["historial_crediticio"] == "con atrasos").astype(int)
tabla["hist_sin"] = (tabla["historial_crediticio"] == "sin historial").astype(int)
tabla["log_monto"] = np.log(tabla["monto_solicitado_usd"]) - np.log(10_000)
tabla["edad_c"] = tabla["edad_anios"] - 38

ATRIBUTOS = ["cuota_ingreso", "hist_atrasos", "hist_sin", "consultas_buro_12m",
             "antiguedad_laboral_anios", "tiene_garantia", "edad_c", "log_monto"]

X = tabla[ATRIBUTOS].to_numpy(dtype=float)
y = tabla["incumplio"].to_numpy(dtype=int)

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
modelo = LogisticRegression(penalty=None, max_iter=5000).fit(X_tr, y_tr)

s = modelo.predict_proba(X_te)[:, 1]   # el puntaje: probabilidad estimada de incumplir
n = len(y_te)
P = int(y_te.sum())
N = n - P

print(f"Entrenamiento: {len(y_tr)}   Prueba: {n}")
print(f"Positivos (incumplimientos): {P}   Negativos: {N}   Prevalencia: {P/n:.4f}")

---

# 1. La matriz de confusión

Fijado un umbral $\tau$, cada solicitud se clasifica como positiva si $s_i \ge \tau$.
Cruzar esa predicción con la clase real da las cuatro celdas:

$$
TP,\ FP,\ FN,\ TN
$$

### Ejercicio 1

Implementar `matriz_confusion`. Devuelve los cuatro conteos, en ese orden.

In [ ]:
def matriz_confusion(y_true: np.ndarray, score: np.ndarray, tau: float) -> tuple[int, int, int, int]:
    """Cuenta las cuatro celdas de la matriz de confusión en el umbral tau.

    Parámetros
    ----------
    y_true : (n,) etiquetas verdaderas en {0, 1}
    score  : (n,) puntajes (probabilidades estimadas de la clase positiva)
    tau    : umbral de decisión

    Devuelve
    --------
    (tp, fp, fn, tn) : los cuatro conteos, como enteros
    """
    # TODO: implementar. pred = (score >= tau); comparar contra y_true y contar
    #       las cuatro combinaciones.
    raise NotImplementedError()

In [ ]:
# VERIFICACIÓN — contra scikit-learn, en tau = 0,5
tp, fp, fn, tn = matriz_confusion(y_te, s, 0.5)
print(f"propia      : TP={tp}  FP={fp}  FN={fn}  TN={tn}")

pred_sk = (s >= 0.5).astype(int)
tn_sk, fp_sk, fn_sk, tp_sk = confusion_matrix(y_te, pred_sk).ravel()
print(f"scikit-learn: TP={tp_sk}  FP={fp_sk}  FN={fn_sk}  TN={tn_sk}")

assert (tp, fp, fn, tn) == (tp_sk, fp_sk, fn_sk, tn_sk)
print("\nCoinciden.")

---

# 2. Métricas sobre un umbral fijo

$$
\text{accuracy}=\frac{TP+TN}{n}
\qquad
\text{precision}=\frac{TP}{TP+FP}
\qquad
\text{recall}=\frac{TP}{TP+FN}
$$

$$
\text{especificidad}=\frac{TN}{TN+FP}
\qquad
F_1=\frac{2\,\text{precision}\cdot\text{recall}}{\text{precision}+\text{recall}}
$$

### Ejercicio 2

Implementar `metricas`, que llama a `matriz_confusion` y devuelve las cinco
cantidades en un diccionario. **Cuidado con la división por cero**: si `TP+FP=0`
la precision no está definida; devolver `0.0` en ese caso (y lo mismo para recall
y F1 si el denominador se anula).

In [ ]:
def metricas(y_true: np.ndarray, score: np.ndarray, tau: float) -> dict:
    """Las cinco métricas de umbral fijo, en el umbral tau.

    Devuelve
    --------
    dict con tp, fp, fn, tn, acc, prec, rec, esp, f1
    """
    # TODO: implementar. Llamar a matriz_confusion y calcular accuracy, precision,
    #       recall, especificidad y F1 a partir de las cuatro celdas. Cubrir la
    #       división por cero devolviendo 0.0.
    raise NotImplementedError()

In [ ]:
# VERIFICACIÓN — contra scikit-learn, en tau = 0,5
m = metricas(y_te, s, 0.5)
pred_sk = (s >= 0.5).astype(int)

print(f"{'métrica':<14}{'propia':>10}{'scikit-learn':>14}")
for nombre, propia, sk in [
    ("accuracy", m["acc"], accuracy_score(y_te, pred_sk)),
    ("precision", m["prec"], precision_score(y_te, pred_sk)),
    ("recall", m["rec"], recall_score(y_te, pred_sk)),
    ("especificidad", m["esp"], recall_score(y_te, pred_sk, pos_label=0)),
    ("F1", m["f1"], f1_score(y_te, pred_sk)),
]:
    print(f"{nombre:<14}{propia:>10.4f}{sk:>14.4f}")
    assert np.isclose(propia, sk)

print(f"\naccuracy del modelo         : {m['acc']:.4f}")
print(f"accuracy del clasificador trivial (predice siempre 'cumple'): {1 - P/n:.4f}")
print("\nTodas coinciden.")

### Para analizar

1. El accuracy del modelo (0,8267) supera al del clasificador trivial (0,7817).
   ¿Alcanza esa diferencia para decir que el modelo es útil? Mirar `recall` antes
   de responder.
2. `especificidad` da un valor alto (0,9520). ¿Qué celda de la matriz sostiene esa
   cifra, y por qué no es la que importa acá?

**Respuesta:**

---

# 3. El barrido: las curvas ROC y precision-recall

Recorriendo $\tau$ de 1 a 0, cada observación pasa a ser positiva exactamente una
vez —cuando $\tau$ cruza su puntaje—, así que no hace falta recalcular la matriz
completa en cada paso: alcanza con ordenar por puntaje y acumular.

$$
\text{ROC}: \bigl(\text{FPR}(\tau),\ \text{TPR}(\tau)\bigr)
\qquad
\text{PR}: \bigl(\text{recall}(\tau),\ \text{precision}(\tau)\bigr)
$$

### Ejercicio 3

Implementar `curva_roc` y `curva_pr`. Las dos ordenan las observaciones por
puntaje **decreciente** y acumulan. `curva_roc` agrega el punto $(0,0)$ al
principio; `curva_pr` no lleva ese punto (el primer punto es la primera
observación clasificada como positiva).

In [ ]:
def curva_roc(y_true: np.ndarray, score: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Curva ROC completa, por barrido del umbral.

    Devuelve
    --------
    (fpr, tpr) : arreglos de igual longitud, con (0, 0) al principio
    """
    # TODO: implementar. Ordenar los índices por score decreciente, acumular
    #       (con np.cumsum) los positivos y los negativos incorporados, y dividir
    #       por P y N respectivamente. Anteponer (0, 0).
    raise NotImplementedError()


def curva_pr(y_true: np.ndarray, score: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Curva precision-recall completa, por barrido del umbral.

    Devuelve
    --------
    (recall, precision) : arreglos de igual longitud
    """
    # TODO: implementar. Mismo orden que curva_roc. precision en el paso k es
    #       (positivos entre los k primeros) / k; recall es (positivos entre los
    #       k primeros) / P.
    raise NotImplementedError()

In [ ]:
# VERIFICACIÓN — propiedades que cualquier curva ROC / PR tiene que cumplir
fpr, tpr = curva_roc(y_te, s)
rec_curva, prec_curva = curva_pr(y_te, s)

assert fpr[0] == 0.0 and tpr[0] == 0.0 and fpr[-1] == 1.0 and tpr[-1] == 1.0
assert np.all(np.diff(fpr) >= 0) and np.all(np.diff(tpr) >= 0), "ROC: no decrecientes"
assert np.all(np.diff(rec_curva) >= 0), "el recall no puede bajar al bajar el umbral"
assert rec_curva[-1] == 1.0
print("Propiedades verificadas: la ROC empieza en (0,0) y termina en (1,1),")
print("y las dos curvas son no decrecientes en su primer eje.")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
axes[0].plot([0, 1], [0, 1], ls="--", color="gray", lw=1.1, label="Al azar")
axes[0].plot(fpr, tpr, color=VIOLETA, lw=2.4, label="Modelo")
for tau, col in [(0.7, TURQUESA), (0.5, NARANJA), (0.2, OLIVA)]:
    mt = metricas(y_te, s, tau)
    axes[0].plot(1 - mt["esp"], mt["rec"], "o", ms=9, color=col, mec="white", mew=1.3)
axes[0].set_xlabel("FPR")
axes[0].set_ylabel("TPR (recall)")
axes[0].set_title("Curva ROC")
axes[0].legend(fontsize=9, frameon=False)

axes[1].axhline(P / n, ls="--", color="gray", lw=1.1, label="Línea base (prevalencia)")
axes[1].plot(rec_curva, prec_curva, color=NARANJA, lw=2.4, label="Modelo")
for tau, col in [(0.7, TURQUESA), (0.5, VIOLETA), (0.2, OLIVA)]:
    mt = metricas(y_te, s, tau)
    axes[1].plot(mt["rec"], mt["prec"], "o", ms=9, color=col, mec="white", mew=1.3)
axes[1].set_xlabel("recall")
axes[1].set_ylabel("precision")
axes[1].set_title("Curva precision-recall")
axes[1].set_ylim(0, 1.02)
axes[1].legend(fontsize=9, frameon=False)
plt.tight_layout()
plt.show()

---

# 4. El AUC por conteo de pares

$$
\widehat{\text{AUC}} = \frac{1}{PN}\sum_{i:\,y_i=1}\sum_{j:\,y_j=0}
\Bigl[\mathbb{1}\{s_i>s_j\} + \tfrac12\,\mathbb{1}\{s_i=s_j\}\Bigr]
$$

### Ejercicio 4

Implementar `auc_pares`: la proporción de pares (positivo, negativo) en que el
positivo recibió mayor puntaje, con medio punto para los empates.

In [ ]:
def auc_pares(y_true: np.ndarray, score: np.ndarray) -> float:
    """AUC como proporción de pares (positivo, negativo) bien ordenados.

    Devuelve
    --------
    float en [0, 1]
    """
    # TODO: implementar. Separar los puntajes de positivos y de negativos, contar
    #       cuántos pares (pos, neg) tienen pos > neg (y medio punto si son
    #       iguales), y dividir por la cantidad total de pares P*N.
    raise NotImplementedError()

In [ ]:
# VERIFICACIÓN — por dos caminos independientes: contra scikit-learn y contra
# el área bajo la propia curva ROC (regla del trapecio).
auc = auc_pares(y_te, s)
auc_sklearn = roc_auc_score(y_te, s)
auc_trapecio = float(np.trapezoid(tpr, fpr))

print(f"AUC por conteo de pares : {auc:.4f}")
print(f"AUC — scikit-learn      : {auc_sklearn:.4f}")
print(f"AUC — trapecios sobre la propia curva ROC: {auc_trapecio:.4f}")

assert np.isclose(auc, auc_sklearn)
assert np.isclose(auc, auc_trapecio, atol=1e-3)
print("\nLas tres coinciden: contar pares y medir el área son la misma cantidad.")

---

# 5. Average precision

$$
\text{AP} = \frac{1}{P}\sum_{k:\ y_{(k)}=1} \text{precision@}k
$$

la precision promediada solo en los pasos donde se recupera un positivo.

### Ejercicio 5

Implementar `average_precision`.

In [ ]:
def average_precision(y_true: np.ndarray, score: np.ndarray) -> float:
    """Average precision: la precision promediada en cada positivo recuperado.

    Devuelve
    --------
    float en [0, 1]
    """
    # TODO: implementar. Mismo orden que curva_pr. Promediar la precision@k
    #       (tp_acum / k) solo en las posiciones k donde yt[k] = 1.
    raise NotImplementedError()

In [ ]:
# VERIFICACIÓN — contra scikit-learn
ap = average_precision(y_te, s)
ap_sklearn = average_precision_score(y_te, s)

print(f"AP propia      : {ap:.4f}")
print(f"AP scikit-learn: {ap_sklearn:.4f}")
print(f"Línea base (prevalencia): {P/n:.4f}")

assert np.isclose(ap, ap_sklearn)
print("\nCoinciden. El modelo concentra los incumplimientos cerca de la cabeza")
print("del ordenamiento con una eficacia bastante por encima de la línea base.")

### Para analizar

El AUC (0,81) y el AP (0,61) describen el mismo modelo y parecen apuntar en
direcciones distintas. Replicar trece veces cada negativo del conjunto de prueba
—sin tocar los puntajes— baja la prevalencia del 21,8 % al 2,1 %. ¿Qué pasa con
cada una de las dos cifras, y por qué? (No hace falta programarlo: alcanza con
razonar a partir de qué usa cada métrica.)

**Respuesta:**

---

# 6. El umbral óptimo bajo tres matrices de costos

$$
\tau^\star = \frac{C_{FP}}{C_{FP}+C_{FN}}
$$

(con $C_{TP}=C_{TN}=0$). No depende del modelo ni de los datos: solo de los costos.

### Ejercicio 6

1. Implementar `tau_optimo(c_fp, c_fn)`.
2. Para cada uno de los tres escenarios de la tabla, calcular el costo total sobre
   el conjunto de prueba con $\tau=0{,}5$ y con $\tau=\tau^\star$, y el ahorro.
   El costo total es $FN\cdot C_{FN} + FP\cdot C_{FP}$.

In [ ]:
def tau_optimo(c_fp: float, c_fn: float) -> float:
    """Umbral que minimiza el costo esperado, con C_TP = C_TN = 0.

    Devuelve
    --------
    float en (0, 1)
    """
    # TODO: implementar la fórmula cerrada.
    raise NotImplementedError()


def costo_total(y_true: np.ndarray, score: np.ndarray, tau: float,
                 c_fp: float, c_fn: float) -> float:
    """Costo total sobre el conjunto, en el umbral tau."""
    # TODO: implementar. Usar metricas(...) para obtener fn y fp en tau, y
    #       devolver fn * c_fn + fp * c_fp.
    raise NotImplementedError()

In [ ]:
ESCENARIOS = [
    ("Cartera de consumo", 300.0, 4000.0),
    ("Costos simétricos", 1000.0, 1000.0),
    ("Revisión costosa", 2400.0, 800.0),
]

filas = []
for nombre, c_fp, c_fn in ESCENARIOS:
    tau_e = tau_optimo(c_fp, c_fn)
    costo_050 = costo_total(y_te, s, 0.5, c_fp, c_fn)
    costo_opt = costo_total(y_te, s, tau_e, c_fp, c_fn)
    filas.append({
        "escenario": nombre, "C_FP": c_fp, "C_FN": c_fn, "tau*": round(tau_e, 4),
        "costo(0,5)": costo_050, "costo(tau*)": costo_opt,
        "ahorro": (costo_050 - costo_opt) / costo_050,
    })

tabla_costos = pd.DataFrame(filas)
print(tabla_costos.to_string(index=False))

In [ ]:
# VERIFICACIÓN — el costo en tau* tiene que estar cerca del mínimo verdadero,
# hallado por búsqueda exhaustiva sobre una grilla fina de umbrales.
taus_grilla = np.linspace(0.001, 0.999, 999)
for nombre, c_fp, c_fn in ESCENARIOS:
    costos_grilla = np.array([costo_total(y_te, s, t, c_fp, c_fn) for t in taus_grilla])
    costo_minimo = costos_grilla.min()
    tau_empirico = taus_grilla[np.argmin(costos_grilla)]
    costo_formula = costo_total(y_te, s, tau_optimo(c_fp, c_fn), c_fp, c_fn)
    print(f"{nombre:<20} tau*(fórmula)={tau_optimo(c_fp, c_fn):.4f}  "
          f"tau*(grilla)={tau_empirico:.4f}   "
          f"costo(fórmula)={costo_formula:,.0f}  costo(mínimo grilla)={costo_minimo:,.0f}")
    assert costo_formula <= costo_minimo * 1.05

print("\nEl costo en el umbral deducido queda a lo sumo un 5 % por encima del")
print("mínimo hallado por grilla. La pequeña distancia entre tau*(fórmula) y")
print("tau*(grilla) es esperable: se discute en el ejercicio DECIDE.")

### DECIDE

En el escenario **Cartera de consumo**, el umbral $\tau^\star=0{,}070$ deja el
accuracy en **0,4708** — peor que el clasificador trivial (0,7817) y que lanzar
una moneda.

1. Un gerente objeta: *"¿cómo va a ser mejor un modelo con menos accuracy?"*.
   Responder con las cifras de costo, no con accuracy.
2. `tau*(fórmula)` y `tau*(grilla)` no coinciden exactamente. ¿A qué se debe la
   diferencia? (La sección 8.4 de las notas nombra dos causas; no hace falta
   desarrollarlas, alcanza con nombrarlas.)
3. ¿Qué información, ausente de `y_te` y `s`, hizo falta para poder calcular
   `tau*` en cualquiera de los tres escenarios?

**Respuesta:**

---

## Cierre

- **La matriz de confusión contiene toda la información sobre un punto de
  operación.** Toda métrica de umbral fijo es una función de sus cuatro celdas.
- **El AUC mide ordenamiento; la average precision, calidad bajo desbalance.**
  Son resúmenes de la misma curva, hechos de dos maneras.
- **El umbral óptimo se deduce de una matriz de costos**, nunca del accuracy.
  0,5 es la elección correcta únicamente cuando los dos errores cuestan lo mismo.

## Para seguir explorando

Sin obligación y fuera del tiempo de clase:

1. **precision@k.** Implementar `precision_at_k(y, score, k)` y graficarla en
   función de $k$. ¿Cuántas de las 100 solicitudes de mayor puntaje terminan en
   incumplimiento?
2. **Multiclase.** Con una matriz de confusión $3\times 3$ armada a mano, calcular
   $F_1$ micro, macro y ponderado, y verificar que micro coincide con accuracy.
3. **Calibración.** La sección 8.4 de las notas menciona que `tau*` supone
   probabilidades calibradas. Graficar, para diez grupos de puntaje equiespaciados,
   la proporción real de incumplimientos contra el puntaje medio del grupo: ¿se
   acerca a la diagonal?